In [7]:
!pip install pandas numpy scikit-learn matplotlib seaborn

Defaulting to user installation because normal site-packages is not writeable


In [9]:
%pip install -U scikit-learn

  Using cached scipy-1.18.0-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ----------------- ---------------------- 3.7/8.2 MB 23.0 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.2 MB 12.4 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.2 MB 12.4 MB/s eta 0:00:01
   -------------------------------- ------- 6.6/8.2 MB 8.2 MB/s eta 0:00:01
   ------------------------------------- -- 7.6/8.2 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 6.5 MB/s eta 0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.18.0-cp313-cp313-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- --------

In [10]:
# Question 7: Healthcare Treatment Cost Prediction Using Lasso Regression

# ---------------------------------------------------------
# 1. Import required libraries
# ---------------------------------------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ---------------------------------------------------------
# 2. Create the dataset
# ---------------------------------------------------------
data = {
    'Patient': ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8'],
    'Age': [25, 35, 45, 55, 60, 30, 50, 40],
    'BP': [115, 125, 140, 155, 165, 120, 150, 135],
    'Sugar_Level': [90, 110, 150, 180, 200, 100, 170, 130],
    'BMI': [22, 25, 29, 32, 35, 24, 31, 27],
    'Previous_Visits': [1, 2, 3, 5, 6, 1, 4, 2],
    'Treatment_Cost': [5000, 8000, 15000, 25000, 32000, 7000, 22000, 12000]
}

df = pd.DataFrame(data)

print("Original Dataset:")
print(df)


# ---------------------------------------------------------
# 3. Identify input features and target variable
# ---------------------------------------------------------

# Input / Independent variables
X = df[['Age', 'BP', 'Sugar_Level', 'BMI', 'Previous_Visits']]

# Output / Dependent variable
y = df['Treatment_Cost']

print("\nInput Features:")
print(X.columns.tolist())

print("\nTarget Variable:")
print(y.name)


# ---------------------------------------------------------
# 4. Train-Test Split
# ---------------------------------------------------------

# 75% training and 25% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

print("\nTraining Data:")
print(X_train)

print("\nTesting Data:")
print(X_test)


# ---------------------------------------------------------
# 5. Feature Scaling
# ---------------------------------------------------------
# Scaling is important for Lasso because L1 regularization
# depends on the magnitude of the coefficients.

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ---------------------------------------------------------
# 6. Test different alpha values
# ---------------------------------------------------------

alpha_values = [0.1, 1.0, 10.0]

results = []

for alpha in alpha_values:

    # Create Lasso model
    model = Lasso(alpha=alpha, max_iter=10000)

    # Train model
    model.fit(X_train_scaled, y_train)

    # Predict test data
    y_pred = model.predict(X_test_scaled)

    # Evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append([
        alpha,
        mae,
        mse,
        rmse,
        r2
    ])


# ---------------------------------------------------------
# 7. Display model comparison
# ---------------------------------------------------------

results_df = pd.DataFrame(
    results,
    columns=['Alpha', 'MAE', 'MSE', 'RMSE', 'R2 Score']
)

print("\nLasso Model Comparison:")
print(results_df.to_string(index=False))


# ---------------------------------------------------------
# 8. Select the best alpha
# ---------------------------------------------------------
# Highest R2 is considered better.

best_alpha = results_df.loc[
    results_df['R2 Score'].idxmax(),
    'Alpha'
]

print("\nBest Alpha:", best_alpha)


# ---------------------------------------------------------
# 9. Train final Lasso model using best alpha
# ---------------------------------------------------------

final_model = Lasso(
    alpha=best_alpha,
    max_iter=10000
)

final_model.fit(X_train_scaled, y_train)


# ---------------------------------------------------------
# 10. Predict treatment cost for test patients
# ---------------------------------------------------------

y_test_pred = final_model.predict(X_test_scaled)

prediction_df = X_test.copy()

prediction_df['Actual Cost'] = y_test.values
prediction_df['Predicted Cost'] = y_test_pred

print("\nTest Dataset Predictions:")
print(prediction_df)


# ---------------------------------------------------------
# 11. Evaluate final model
# ---------------------------------------------------------

mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

print("\nFinal Model Evaluation:")
print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R2 Score:", r2)


# ---------------------------------------------------------
# 12. Predict treatment cost for a new patient
# ---------------------------------------------------------

new_patient = pd.DataFrame({
    'Age': [48],
    'BP': [145],
    'Sugar_Level': [160],
    'BMI': [30],
    'Previous_Visits': [4]
})

# Scale new patient using the same scaler
new_patient_scaled = scaler.transform(new_patient)

# Prediction
new_prediction = final_model.predict(new_patient_scaled)

print("\nNew Patient:")
print(new_patient)

print("\nPredicted Treatment Cost for New Patient:")
print("₹", round(new_prediction[0], 2))


# ---------------------------------------------------------
# 13. Print Lasso coefficients
# ---------------------------------------------------------

coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': final_model.coef_
})

print("\nLasso Model Coefficients:")
print(coefficients)


# ---------------------------------------------------------
# 14. Identify important features
# ---------------------------------------------------------

print("\nFeature Importance Based on Lasso Coefficients:")

for feature, coefficient in zip(X.columns, final_model.coef_):

    if coefficient != 0:
        print(
            feature,
            "-> Important",
            "(Coefficient =", round(coefficient, 2), ")"
        )
    else:
        print(
            feature,
            "-> Removed by Lasso"
        )


# ---------------------------------------------------------
# 15. Final interpretation
# ---------------------------------------------------------

print("\nInterpretation:")
print("Lasso Regression uses L1 regularization to reduce")
print("the effect of less important features.")
print("Features with coefficients close to or equal to zero")
print("have less influence on treatment cost.")
print("Features with larger absolute coefficients have")
print("greater influence on the predicted treatment cost.")

Original Dataset:
  Patient  Age   BP  Sugar_Level  BMI  Previous_Visits  Treatment_Cost
0      P1   25  115           90   22                1            5000
1      P2   35  125          110   25                2            8000
2      P3   45  140          150   29                3           15000
3      P4   55  155          180   32                5           25000
4      P5   60  165          200   35                6           32000
5      P6   30  120          100   24                1            7000
6      P7   50  150          170   31                4           22000
7      P8   40  135          130   27                2           12000

Input Features:
['Age', 'BP', 'Sugar_Level', 'BMI', 'Previous_Visits']

Target Variable:
Treatment_Cost

Training Data:
   Age   BP  Sugar_Level  BMI  Previous_Visits
0   25  115           90   22                1
7   40  135          130   27                2
2   45  140          150   29                3
4   60  165          200   35     

Why use Lasso Regression?

Lasso Regression is suitable for this healthcare problem because the hospital has several medical predictors and wants to identify which variables are most useful for predicting treatment cost. Lasso applies L1 regularization, which can shrink some coefficients exactly to zero. Therefore, it performs both prediction and feature selection, helping identify the most important medical factors while reducing the influence of less useful predictors.

Interpretation of coefficients:

Positive coefficient → an increase in that feature tends to increase predicted treatment cost.
Negative coefficient → an increase in that feature tends to decrease predicted treatment cost.
Coefficient = 0 → Lasso has effectively removed that feature from the model.
Larger absolute coefficient → stronger influence on the prediction, assuming the features have been standardized.

Note: Because this dataset contains only 8 patients, the train/test results can vary substantially depending on the split. The metrics should therefore be interpreted as an exercise result rather than as evidence that the model is clinically reliable.